# 14.16 能保證多一輪，loss 就下降嗎？


假設同一留出集的 loss 在1到5輪依序是1.2、0.8、0.6、0.7、0.9。若部署目標就是這批任務，第三輪目前最好，繼續算可能浪費工作。這不是違反循環架構：共享區塊沒有被定義成「每次都朝正解下降」的最佳化器。

有幾種辦法能**促進**改善。逐輪監督讓每輪都對答案負責；也可以在總目標加上 `λ*max(0,L_(r+1)-L_r)`，懲罰後一輪比前一輪差的情況。λ 是權重，這是軟限制：最佳化器在多個目標、題目及參數之間取捨，仍可能保留違反項，對新題更沒有逐題保證。

下面用人工 loss 檢查一個陷阱。只看「後輪比前輪差多少」的懲罰，模型甚至可以把早一輪變差，使差額縮小；因此還要監督各輪本身的品質。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
def penalty(before, after):
    return max(0.0, after - before)


for before, after in [(0.2, 0.4), (0.5, 0.4)]:
    print("早輪", before, "後輪", after, "懲罰", penalty(before, after))

兩組的後輪都是0.4；早輪從0.2惡化成0.5，懲罰卻從0.2變0。把懲罰變小不等於每輪能力提高。若加上各輪 CE，這條退路會付出其他代價，但仍是要實際平衡的訓練目標。

要談「保證」，須先說保證什麼：訓練集平均 loss、每個訓練樣本、任意新題，還是正確率？這些要求不同。交叉熵下降也不一定讓 argmax 改成正解；正確率可以不變。限制狀態更新收斂，只保證靠近某個固定狀態時，也還沒證明那個狀態答對。

研究中有多深度監督、遞迴 loss 正則與自適應停止等方向；例如 LoopUS 把訓練正則與停止設計放在一起研究。這類軟目標提供工程工具，不能改寫成所有新題的單調定理。一般部署可先在 validation 選定固定輪數；想逐題早停，還要有使用時拿得到的訊號。

練習：若第三輪 loss 最低但第四輪正確率最高，哪輪最好？先看產品目標；若任務按最後答案對錯驗收，不能只照 loss 排序。報告兩條曲線並保留差異，比要求它們一致更有用。

研究例：[LoopUS](https://arxiv.org/abs/2605.11011)。
